# NB01c — Manual Frame Picker

Pilih video, putar/geser frame, lalu klik **Ambil frame ini**.

In [14]:
import csv, os, shutil
from datetime import datetime
from pathlib import Path
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from pipeline_common import build_roi_crop, load_roi_config

RISET_ROOT = Path(os.environ.get("RISET_PROJECT_ROOT", Path.cwd())).resolve()
if RISET_ROOT.name not in {"Riset-HampirMatang", "NewStyle"}:
    raise RuntimeError("Set RISET_PROJECT_ROOT atau jalankan dari root proyek.")
OUTPUT_ROOT = RISET_ROOT / "output"
VIDEO_ROOT = OUTPUT_ROOT / "videos"
FRAME_ROOT = OUTPUT_ROOT / "curated_frames"
CROP_ROOT = OUTPUT_ROOT / "curated_crops"
MANIFEST_ROOT = OUTPUT_ROOT / "manifests"
ROI_CONFIG = load_roi_config(OUTPUT_ROOT / "roi")
MIN_GAP_FRAMES, HASH_DISTANCE = 30, 4
selected_rows = []
FRAME_ROOT.mkdir(parents=True, exist_ok=True)
CROP_ROOT.mkdir(parents=True, exist_ok=True)


def d_hash(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA).astype(int)
    return sum(
        1 << i
        for i, value in enumerate((resized[:, 1:] >= resized[:, :-1]).flatten())
        if value
    )


def open_frame(index):
    capture = cv2.VideoCapture(video_dropdown.value)
    capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame = capture.read()
    capture.release()
    return frame if ok else None


video_dropdown = widgets.Dropdown(
    options=[(p.name, str(p)) for p in sorted(VIDEO_ROOT.glob("*.mp4"))],
    description="Video:",
    layout=widgets.Layout(width="95%"),
)
frame_slider = widgets.IntSlider(
    min=0,
    max=1,
    step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)
play_control = widgets.Play(
    value=0, min=0, max=1, step=1, interval=80, description="Putar"
)
widgets.jslink((play_control, "value"), (frame_slider, "value"))
take_button = widgets.Button(description="Ambil frame ini", button_style="success")
rare_button = widgets.Button(description="Ambil rare case", button_style="warning")
preview, status = widgets.Output(), widgets.Output()


def refresh_video(change=None):
    capture = cv2.VideoCapture(video_dropdown.value)
    frame_slider.max = max(0, int(capture.get(cv2.CAP_PROP_FRAME_COUNT)) - 1)
    play_control.max = frame_slider.max
    capture.release()
    show_frame()


def show_frame(change=None):
    frame = open_frame(frame_slider.value)
    with preview:
        clear_output(wait=True)
        if frame is not None:
            display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))


def take_frame(button, force=False):
    video_path, frame_index = Path(video_dropdown.value), int(frame_slider.value)
    frame = open_frame(frame_index)
    if frame is None:
        raise RuntimeError(f"frame {frame_index} tidak bisa dibaca")
    relative_video = str(video_path.relative_to(RISET_ROOT))
    frame_hash = d_hash(frame)
    previous = [row for row in selected_rows if row["video_path"] == relative_video]
    too_close = any(
        abs(frame_index - row["frame_index"]) < MIN_GAP_FRAMES for row in previous
    )
    too_similar = any(
        (frame_hash ^ row["frame_hash"]).bit_count() <= HASH_DISTANCE
        for row in previous
    )
    if too_close and not force:
        message = f"ditolak: jarak < {MIN_GAP_FRAMES} frame"
    elif too_similar and not force:
        message = f"ditolak: dHash <= {HASH_DISTANCE}"
    else:
        camera_id = video_path.stem.split("_")[1]
        frame_path = FRAME_ROOT / video_path.stem / f"frame_{frame_index:06d}.png"
        frame_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(frame_path), frame)
        with Image.open(frame_path) as image:
            crop, _, _, _ = build_roi_crop(
                image.convert("RGB"), ROI_CONFIG.get(camera_id)
            )
        crop.save(CROP_ROOT / f"{video_path.stem}_frame_{frame_index:06d}.png")
        selected_rows.append(
            {
                "camera_id": camera_id,
                "video_path": relative_video,
                "frame_path": str(frame_path.relative_to(RISET_ROOT)),
                "frame_index": frame_index,
                "frame_hash": frame_hash,
                "is_duplicate": False,
                "selection_mode": "manual_override" if force else "manual",
            }
        )
        message = f"disimpan: {len(selected_rows)} frame"
    with status:
        clear_output(wait=True)
        display(message)


video_dropdown.observe(refresh_video, names="value")
frame_slider.observe(show_frame, names="value")
take_button.on_click(lambda button: take_frame(button, force=False))
rare_button.on_click(lambda button: take_frame(button, force=True))
refresh_video()
display(
    widgets.VBox(
        [
            video_dropdown,
            widgets.HBox([play_control, frame_slider]),
            widgets.HBox([take_button, rare_button]),
            preview,
            status,
        ]
    )
)

## Simpan manifest curated

Jalankan setelah selesai memilih frame.

In [15]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_root = MANIFEST_ROOT / f"curated_backup_{timestamp}"
for name in ("curated_frames_manifest.csv", "curated_dedup_manifest.csv"):
    existing = MANIFEST_ROOT / name
    if existing.exists():
        backup_root.mkdir(parents=True, exist_ok=True)
        shutil.copy2(existing, backup_root / name)
frames_path, dedup_path = (
    MANIFEST_ROOT / "curated_frames_manifest.csv",
    MANIFEST_ROOT / "curated_dedup_manifest.csv",
)
existing_rows = []
if dedup_path.exists():
    with dedup_path.open(encoding="utf-8") as handle:
        existing_rows = list(csv.DictReader(handle))
for row in existing_rows:
    row.setdefault("selection_mode", "existing")
merged_rows = {(row["video_path"], row["frame_path"]): row for row in existing_rows}
for row in selected_rows:
    merged_rows.setdefault((row["video_path"], row["frame_path"]), row)
all_rows = list(merged_rows.values())
with frames_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["video_path", "frame_path"])
    writer.writeheader()
    writer.writerows(
        {key: row[key] for key in ("video_path", "frame_path")} for row in all_rows
    )
with dedup_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "video_path",
            "frame_path",
            "frame_hash",
            "is_duplicate",
            "selection_mode",
        ],
    )
    writer.writeheader()
    writer.writerows(
        {
            key: row[key]
            for key in (
                "video_path",
                "frame_path",
                "frame_hash",
                "is_duplicate",
                "selection_mode",
            )
        }
        for row in all_rows
    )
display(
    {
        "new_selected": len(selected_rows),
        "existing_kept": len(existing_rows),
        "total_saved": len(all_rows),
        "frames_manifest": str(frames_path),
        "dedup_manifest": str(dedup_path),
        "backup": str(backup_root) if backup_root.exists() else None,
    }
)

{'new_selected': 0,
 'existing_kept': 528,
 'total_saved': 528,
 'frames_manifest': 'D:\\Jere\\NewStyle\\output\\manifests\\curated_frames_manifest.csv',
 'dedup_manifest': 'D:\\Jere\\NewStyle\\output\\manifests\\curated_dedup_manifest.csv',
 'backup': 'D:\\Jere\\NewStyle\\output\\manifests\\curated_backup_20261103_011629'}